**Models:** four variation of logistic regression.  

**Scores and Best Parameters:**  
Logistic Regression: 0.6847212165097756.  
Best Paramters: {'logistic__C': np.float64(0.1)}

Logistic Regression with PCA: 0.6851556842867488.  
Best Paramters: {'logistic__C': np.float64(0.1), 'pca__n_components': np.int64(20)}

Logistic Regression with only stats difference: 0.6728457639391745.   
Best Paramters: {'logistic__C': np.float64(0.01)}

Logistic Regression without elo features: 0.6547429398986242   
Best Paramters: {'logistic__C': np.float64(0.001)}

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Preparation

In [2]:
data = pd.read_csv('match_data_300_tourns_modified.csv')
data.head()

,player1,player2,best_of,player1_elo,player2_elo,elo_match_win_rate,elo_frame_win_rate,p1_matches_played,p1_matches_won,p1_frames_played,...,p2_frames_played_1_year,p2_frames_won_1_year,p2_frames_played_3_years,p2_frames_won_3_years,score1,score2,match_result,win_percentage,tournament_id,date
0,Mark Allen,Ricky Walden,7,1580,1418,0.709992,0.599888,501,313,3345,...,45,25,1032,557,0,4,1.0,0.000000,1063,1.415318e+18
1,Stephen Maguire,Judd Trump,7,1563,1551,0.516401,0.507499,663,444,4995,...,122,79,1344,782,1,4,1.0,0.200000,1063,1.415059e+18
2,Mark Selby,Steve Davis,7,1592,1246,0.878716,0.703704,736,495,5330,...,8,2,581,286,4,1,0.0,0.800000,1063,1.415059e+18
3,Neil Robertson,Ali Carter,7,1546,1544,0.502734,0.501250,621,402,4581,...,11,5,876,499,4,0,0.0,1.000000,1063,1.415318e+18
4,Stuart Bingham,Ronnie O'Sullivan,7,1488,1663,0.275106,0.392337,744,462,5592,...,71,46,674,431,2,4,1.0,0.333333,1063,1.415146e+18


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 34521 entries, 0 to 34520
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   player1                   34521 non-null  object 
 1   player2                   34521 non-null  object 
 2   best_of                   34521 non-null  int64  
 3   player1_elo               34521 non-null  int64  
 4   player2_elo               34521 non-null  int64  
 5   elo_match_win_rate        34521 non-null  float64
 6   elo_frame_win_rate        34521 non-null  float64
 7   p1_matches_played         34521 non-null  int64  
 8   p1_matches_won            34521 non-null  int64  
 9   p1_frames_played          34521 non-null  int64  
 10  p1_frames_won             34521 non-null  int64  
 11  p2_matches_played         34521 non-null  int64  
 12  p2_matches_won            34521 non-null  int64  
 13  p2_frames_played          34521 non-null  int64  
 14  p2_fra

In [4]:
#Train test split
from sklearn.model_selection import train_test_split
data_train, data_test = train_test_split(data, 
                                        test_size = 0.2,
                                        shuffle=False)


In [5]:
#Create predictors and targets for training and cross-validation set
y_train = data_train['match_result']
y_test = data_test['match_result']

#To get the predictor, we exclude players' names (Strings), date, tournament ids and match results.
X_train = data_train.drop(['match_result', 'win_percentage', 'score1', 'score2', 'player1', 'player2', 'tournament_id', 'date'],
                           axis = 1)
X_test = data_test.drop(['match_result', 'win_percentage', 'score1', 'score2', 'player1', 'player2', 'tournament_id', 'date'],
                           axis = 1)

***

## Performance Metrics
Because the two classes in the target are symmetric (swapping player1 and player2 will exchange positive and negative but still represents the same match), we won't consider metrics such as presision, specificity and sensitivity since they are the same as accuracy score. We will only consider accuracy score.

In [6]:
#Import metrics
from sklearn.metrics import accuracy_score

In [7]:
#Create dictionaries to store the metrics.
accuracy_scores = {}

#Create a list to store all the models we consider.
models = {}

In [8]:
def print_avg_cv_metrics(model, model_name):
    """
    Given a model, computes and stores accuracy scores on the test set.
    """

    #Create an empty array to store the scores.
    print('Currently working on ' + model_name + '.')

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)

    score = accuracy_score(y_test, y_pred)

    print('The accuracy score of ' + model_name + ' is:', score)
    
    #Record the scores
    accuracy_scores[model_name] = score

    models[model_name] = model


***

### Logistic Regression with all features

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

In [10]:
scaler = StandardScaler()
log_reg = LogisticRegression(max_iter=10000)

log_reg = Pipeline([('scale', scaler),
                    ('logistic', log_reg)])

param_grid = {
    "logistic__C": np.logspace(-3, 3, 7),
}


grid_search1 = GridSearchCV(log_reg,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)
grid_search1.fit(X_train, y_train.values)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scale', StandardScaler()),
                                       ('logistic',
                                        LogisticRegression(max_iter=10000))]),
             param_grid={'logistic__C': array([1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02, 1.e+03])},
             scoring='accuracy')

In [11]:
# Print the parameters with best performance in the cross_validations and the corresponding score.
print(grid_search1.best_params_)
print(grid_search1.best_score_)

{'logistic__C': np.float64(0.1)}
0.691989911715382


In [12]:
# Use the model to make prediction on the test set and print the score.
model1 = grid_search1.best_estimator_
print_avg_cv_metrics(model1, 'Logistic Regression')

Currently working on Logistic Regression.
The accuracy score of Logistic Regression is: 0.6847212165097756


In [13]:
# Print the coefficients of logistic regression and rank the features. 
print(model1.named_steps['logistic'].coef_)
indices = np.argsort(np.abs(model1.named_steps['logistic'].coef_))
indices = np.flip(indices)
print(X_train.columns.values[indices])


[[ 0.01656578 -0.29146545  0.35653012 -1.43084985  0.72374118  0.10942592
  -0.07463676  0.1148963  -0.10196924 -0.20415153  0.13404851 -0.14004155
   0.16948267 -0.94923704  0.92259605 -0.64551772  0.49447054  0.89468244
  -0.84638548  0.61467126 -0.58083776]]
[['elo_match_win_rate' 'p1_frames_played_1_year' 'p1_frames_won_1_year'
  'p2_frames_played_1_year' 'p2_frames_won_1_year' 'elo_frame_win_rate'
  'p1_frames_played_3_years' 'p2_frames_played_3_years'
  'p2_frames_won_3_years' 'p1_frames_won_3_years' 'player2_elo'
  'player1_elo' 'p2_matches_played' 'p2_frames_won' 'p2_frames_played'
  'p2_matches_won' 'p1_frames_played' 'p1_matches_played' 'p1_frames_won'
  'p1_matches_won' 'best_of']]


We rank the size of coefficients of the logistic regression model. Top 4 are all elo rating related features created by our elo rating system.

***

### Logistic Regression with PCA

In [14]:
from sklearn.decomposition import PCA

In [15]:
scaler = StandardScaler()
pca = PCA()
logistic = LogisticRegression(max_iter=10000)

log_reg_with_pca = Pipeline([('scale', scaler),
                             ('pca', pca),
                    ('logistic', logistic)])

param_grid = {
    "logistic__C": np.logspace(-3, 3, 7),
    "pca__n_components": np.linspace(5, 20, 4).astype(int)
}


grid_search2 = GridSearchCV(log_reg_with_pca,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)
grid_search2.fit(X_train, y_train.values)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scale', StandardScaler()),
                                       ('pca', PCA()),
                                       ('logistic',
                                        LogisticRegression(max_iter=10000))]),
             param_grid={'logistic__C': array([1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02, 1.e+03]),
                         'pca__n_components': array([ 5, 10, 15, 20])},
             scoring='accuracy')

In [16]:
# Print the parameters with best performance in the cross_validations and the corresponding score.
print(grid_search2.best_params_)
print(grid_search2.best_score_)

{'logistic__C': np.float64(0.1), 'pca__n_components': np.int64(20)}
0.6917726319388751


In [17]:
# Use the model to make prediction on the test set and print the score.
model2 = grid_search2.best_estimator_
print_avg_cv_metrics(model2, 'Logistic Regression with PCA')

Currently working on Logistic Regression with PCA.
The accuracy score of Logistic Regression with PCA is: 0.6851556842867488


***

### Logistic Regression with only stats difference

In this model, we only consider the difference between two players stats as the features.

In [18]:
df = data.copy()
df['elo_diff'] = df['player2_elo'] - df['player1_elo']
df['match_played_diff'] = df['p2_matches_played']-df['p1_matches_played']
df['frames_played_diff'] = df['p2_frames_played']-df['p1_frames_played']
df['frames_played_3_diff'] = df['p2_frames_played_3_years']-df['p1_frames_played_3_years']
df['frames_played_1_diff'] = df['p2_frames_played_1_year']-df['p1_frames_played_1_year']

features = ['elo_diff', 'match_played_diff', 'frames_played_diff', 'frames_played_3_diff', 'frames_played_1_diff']

df_train, df_test = train_test_split(df, 
                                        test_size = 0.2,
                                        shuffle=False)

X_train, X_test = df_train[features], df_test[features]

In [19]:
scaler = StandardScaler()
logistic = LogisticRegression(max_iter=10000)

log_reg2 = Pipeline([('scale', scaler),
                    ('logistic', logistic)])

param_grid = {
    "logistic__C": np.logspace(-3, 3, 7),
}


grid_search3 = GridSearchCV(log_reg2,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)
grid_search3.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scale', StandardScaler()),
                                       ('logistic',
                                        LogisticRegression(max_iter=10000))]),
             param_grid={'logistic__C': array([1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02, 1.e+03])},
             scoring='accuracy')

In [20]:
# Print the parameters with best performance in the cross_validations and the corresponding score.
print(grid_search3.best_params_)
print(grid_search3.best_score_)

{'logistic__C': np.float64(0.001)}
0.686666593245834


In [21]:
# Use the model to make prediction on the test set and print the score.
model3 = grid_search3.best_estimator_
print_avg_cv_metrics(model3, 'Logistic Regression with only stats difference')

Currently working on Logistic Regression with only stats difference.
The accuracy score of Logistic Regression with only stats difference is: 0.6728457639391745


***

### Logistic Regression without elo features

In [22]:
#Create predictors and targets for training and cross-validation set
y_train = data_train['match_result']
y_test = data_test['match_result']

#To get the predictor, we exclude players' names (Strings), date, tournament ids and match results, and elo features.
X_train = data_train.drop(['match_result', 'win_percentage', 'score1', 'score2',
                            'player1', 'player2', 'tournament_id', 'date',
                            'player1_elo', 'player2_elo', 'elo_match_win_rate', 'elo_frame_win_rate'],
                           axis = 1)
X_test = data_test.drop(['match_result', 'win_percentage', 'score1', 'score2',
                            'player1', 'player2', 'tournament_id', 'date',
                            'player1_elo', 'player2_elo', 'elo_match_win_rate', 'elo_frame_win_rate'],
                           axis = 1)



In [23]:
scaler = StandardScaler()
log_reg3 = LogisticRegression(max_iter=10000)

log_reg3 = Pipeline([('scale', scaler),
                    ('logistic', log_reg)])

param_grid = {
    "logistic__C": np.logspace(-3, 3, 7),
}


grid_search4 = GridSearchCV(log_reg,
                           param_grid=param_grid,
                           scoring='accuracy',
                           cv=5)
grid_search4.fit(X_train, y_train.values)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scale', StandardScaler()),
                                       ('logistic',
                                        LogisticRegression(max_iter=10000))]),
             param_grid={'logistic__C': array([1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02, 1.e+03])},
             scoring='accuracy')

In [24]:
# Print the parameters with best performance in the cross_validations and the corresponding score.
print(grid_search4.best_params_)
print(grid_search4.best_score_)

{'logistic__C': np.float64(0.001)}
0.6776498070146526


In [25]:
# Use the model to make prediction on the test set and print the score.
model4 = grid_search3.best_estimator_
print_avg_cv_metrics(model4, 'Logistic Regression without elo features')

Currently working on Logistic Regression without elo features.
The accuracy score of Logistic Regression without elo features is: 0.6547429398986242


***

In [26]:
# Print the scores
print(accuracy_scores)

{'Logistic Regression': 0.6847212165097756, 'Logistic Regression with PCA': 0.6851556842867488, 'Logistic Regression with only stats difference': 0.6728457639391745, 'Logistic Regression without elo features': 0.6547429398986242}
